# 🚢 Titanic Survival Analysis — Data|Driven Intelligence

**Assignment:** Machine Learning from Disaster (Kaggle Titanic)  
**Approach:** NumPy + Pandas only — No AutoML, No Sklearn  
**Objective:** Build interpretable survival prediction logic from first principles

|||

## 📦 Imports & Configuration


In [1]:
import csv
import numpy as np
import pandas as pd
import urllib.request
import warnings
warnings.filterwarnings('ignore')

# random seed for reproducibility
np.random.seed(42)

print("✓ Environment ready")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


✓ Environment ready
NumPy version: 1.26.4
Pandas version: 2.3.3


|||
## 📥 Data Loading (Manual CSV Parse — No Pandas Initially)

We download the train dataset from GitHub (Kaggle mirror) and load it using Python's native `csv` module.


In [2]:
# download Titanic train.csv
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
urllib.request.urlretrieve(url, "titanic_train.csv")

# load raw rows using csv.DictReader (pure Python)
raw_rows = []
with open("titanic_train.csv", "r", encoding="utf|8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        raw_rows.append(row)

print(f"✓ Loaded {len(raw_rows)} passengers")
print(f"Columns: {list(raw_rows[0].keys())}")
print(f"\nSample row:\n{raw_rows[0]}")


✓ Loaded 891 passengers
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Sample row:
{'PassengerId': '1', 'Survived': '0', 'Pclass': '3', 'Name': 'Braund, Mr. Owen Harris', 'Sex': 'male', 'Age': '22', 'SibSp': '1', 'Parch': '0', 'Ticket': 'A/5 21171', 'Fare': '7.25', 'Cabin': '', 'Embarked': 'S'}


|||
# Part 1 — Raw Data Exploration (Lists + NumPy Only)

## 1.1 Age → Python List + Manual Missing Value Removal


In [3]:
# extract Age as raw Python list
age_raw_list = [row['Age'] for row in raw_rows]
print(f"Total Age entries (raw): {len(age_raw_list)}")

# manually remove missing values (empty strings)
age_clean_list = [float(a) for a in age_raw_list if a.strip() != '']
print(f"Valid Age entries: {len(age_clean_list)}")
print(f"Missing ages: {len(age_raw_list) | len(age_clean_list)}")
print(f"\nFirst 10 ages: {age_clean_list[:10]}")


Total Age entries (raw): 891
Valid Age entries: 714
Missing ages: 177

First 10 ages: [22.0, 38.0, 26.0, 35.0, 35.0, 54.0, 2.0, 27.0, 14.0, 4.0]


## 1.2 Age Statistics (NumPy)

In [4]:
# convert to NumPy array
age_array = np.array(age_clean_list)

# compute statistics
mean_age = np.mean(age_array)
median_age = np.median(age_array)
std_age = np.std(age_array)
min_age = np.min(age_array)
max_age = np.max(age_array)

print("=" * 40)
print("AGE STATISTICS (NumPy)")
print("=" * 40)
print(f"Mean Age       : {mean_age:.2f} years")
print(f"Median Age     : {median_age:.2f} years")
print(f"Std Deviation  : {std_age:.2f} years")
print(f"Min Age        : {min_age:.2f} years")
print(f"Max Age        : {max_age:.2f} years")
print("=" * 40)


AGE STATISTICS (NumPy)
Mean Age       : 29.70 years
Median Age     : 28.00 years
Std Deviation  : 14.52 years
Min Age        : 0.42 years
Max Age        : 80.00 years


## 1.3 Fare Analysis — Top 10% vs Bottom 10%

In [5]:
# build NumPy arrays of Fare and Survived (complete data)
fare_array = np.array([float(row['Fare']) for row in raw_rows])
survived_array = np.array([int(row['Survived']) for row in raw_rows])

# compute 90th and 10th percentiles
p90 = np.percentile(fare_array, 90)
p10 = np.percentile(fare_array, 10)

# boolean masks
top10_mask = fare_array >= p90
bot10_mask = fare_array <= p10

# survival rates
top10_survival = survived_array[top10_mask].mean()
bot10_survival = survived_array[bot10_mask].mean()

print("=" * 50)
print("FARE ANALYSIS — TOP 10% vs BOTTOM 10%")
print("=" * 50)
print(f"90th Percentile : £{p90:.2f}")
print(f"10th Percentile : £{p10:.2f}")
print(f"\nTop 10% count  : {top10_mask.sum()} passengers")
print(f"Bot 10% count  : {bot10_mask.sum()} passengers")
print(f"\nTop 10% Survival Rate : {top10_survival:.2%}")
print(f"Bot 10% Survival Rate : {bot10_survival:.2%}")
print(f"\nSurvival Ratio (Top/Bot) : {top10_survival/bot10_survival:.2f}x")
print("=" * 50)


FARE ANALYSIS — TOP 10% vs BOTTOM 10%
90th Percentile : £77.96
10th Percentile : £7.55

Top 10% count  : 90 passengers
Bot 10% count  : 92 passengers

Top 10% Survival Rate : 76.67%
Bot 10% Survival Rate : 14.13%

Survival Ratio (Top/Bot) : 5.43x


## 1.4 Age|Group Survival Comparison (NumPy Boolean Indexing)

In [6]:
# build parallel arrays: age and survival (only valid age rows)
age_surv_pairs = [(float(row['Age']), int(row['Survived'])) 
                  for row in raw_rows if row['Age'].strip() != '']

age_arr = np.array([p[0] for p in age_surv_pairs])
surv_arr = np.array([p[1] for p in age_surv_pairs])

# define age groups
child_mask = age_arr < 15
adult_mask = (age_arr >= 15) & (age_arr <= 60)
senior_mask = age_arr > 60

# compute survival rates per group
child_survival = surv_arr[child_mask].mean()
adult_survival = surv_arr[adult_mask].mean()
senior_survival = surv_arr[senior_mask].mean()

print("=" * 55)
print("AGE|GROUP SURVIVAL RATES (NumPy Filtering)")
print("=" * 55)
print(f"Children  (< 15)  : {child_mask.sum():>4} passengers | Survival: {child_survival:.2%}")
print(f"Adults    (15|60) : {adult_mask.sum():>4} passengers | Survival: {adult_survival:.2%}")
print(f"Seniors   (> 60)  : {senior_mask.sum():>4} passengers | Survival: {senior_survival:.2%}")
print("=" * 55)
print("\nObservation: Children had highest survival (women & children first protocol)")


AGE|GROUP SURVIVAL RATES (NumPy Filtering)
Children  (< 15)  :   78 passengers | Survival: 57.69%
Adults    (15|60) :  614 passengers | Survival: 39.09%
Seniors   (> 60)  :   22 passengers | Survival: 22.73%

Observation: Children had highest survival (women & children first protocol)


## 1.5 Insight Q1: Is Age Linearly Related to Survival?

**Method:** Pearson correlation coefficient using `np.corrcoef()`


In [7]:
# compute Pearson correlation
corr_matrix = np.corrcoef(age_arr, surv_arr)
pearson_r = corr_matrix[0, 1]
r_squared = pearson_r ** 2

print("=" * 60)
print("LINEARITY TEST: Age vs Survival")
print("=" * 60)
print(f"Pearson Correlation (r) : {pearson_r:.4f}")
print(f"R² (variance explained) : {r_squared:.4f} ({r_squared*100:.2f}%)")
print("=" * 60)

# interpretation
if abs(pearson_r) < 0.1:
    strength = "negligible"
elif abs(pearson_r) < 0.3:
    strength = "weak"
elif abs(pearson_r) < 0.5:
    strength = "moderate"
else:
    strength = "strong"

print(f"\nInterpretation: {strength.upper()} linear relationship")
print()
print("ANSWER: Age is NOT linearly related to survival.")
print("Justification:")
print(f"  • Pearson r = {pearson_r:.4f} indicates almost no linear trend")
print(f"  • Only {r_squared*100:.2f}% of survival variance explained by age")
print("  • Non|linear patterns exist: children prioritized, seniors disadvantaged")
print("  • Age alone is a poor linear predictor")


LINEARITY TEST: Age vs Survival
Pearson Correlation (r) : |0.0772
R² (variance explained) : 0.0060 (0.60%)

Interpretation: NEGLIGIBLE linear relationship

ANSWER: Age is NOT linearly related to survival.
Justification:
  • Pearson r = |0.0772 indicates almost no linear trend
  • Only 0.60% of survival variance explained by age
  • Non|linear patterns exist: children prioritized, seniors disadvantaged
  • Age alone is a poor linear predictor


|||
# Part 2 — Advanced Pandas Engineering

## 2.1 Load Data with Pandas


In [8]:
df = pd.read_csv("titanic_train.csv")
print(f"Shape: {df.shape}")
print(f"\nInfo:")
print(df.info())
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])


Shape: (891, 12)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non|Null Count  Dtype  
|||  ||||||       ||||||||||||||  |||||  
 0   PassengerId  891 non|null    int64  
 1   Survived     891 non|null    int64  
 2   Pclass       891 non|null    int64  
 3   Name         891 non|null    object 
 4   Sex          891 non|null    object 
 5   Age          714 non|null    float64
 6   SibSp        891 non|null    int64  
 7   Parch        891 non|null    int64  
 8   Ticket       891 non|null    object 
 9   Fare         891 non|null    float64
 10  Cabin        204 non|null    object 
 11  Embarked     889 non|null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None

Missing values:
Age         177
Cabin       687
Embarked      2
dtype: int64


## 2.2 Handle Missing Values

**Strategy:**
| **Age** → median imputation **per Pclass** (class|specific demographics)
| **Embarked** → mode imputation (most common port)
| **Cabin** → drop column (77% missing, not recoverable)


In [9]:
# Age → median by Pclass (class 1 passengers were generally older)
df['Age'] = df.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))

# Embarked → mode
embarked_mode = df['Embarked'].mode()[0]
df['Embarked'].fillna(embarked_mode, inplace=True)

# drop Cabin (too sparse)
df.drop(columns=['Cabin'], inplace=True)

print("✓ Missing values handled")
print(f"\nClass|wise Age medians used:")
print(df.groupby('Pclass')['Age'].median())
print(f"\nEmbarked mode: {embarked_mode}")
print(f"\nRemaining missing values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None — dataset is clean!")


✓ Missing values handled

Class|wise Age medians used:
Pclass
1    37.0
2    29.0
3    24.0
Name: Age, dtype: float64

Embarked mode: S

Remaining missing values:
None — dataset is clean!


## 2.3 Feature Engineering

We create the following derived features:

| Feature | Formula | Rationale |
|||||||||||||||||||||||||||||||||
| `FamilySize` | SibSp + Parch + 1 | Total family members on board |
| `IsAlone` | 1 if FamilySize == 1 else 0 | Solo travelers behave differently |
| `FarePerPerson` | Fare / FamilySize | Normalize fare to individual spending |
| `AgeGroup` | Child / Adult / Senior | Capture non|linear age effects |
| `FareGroup` | Low / Medium / High | Wealth tier classification |


In [10]:
# FamilySize
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# IsAlone (binary)
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# FarePerPerson
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

# AgeGroup (categorical bins)
df['AgeGroup'] = pd.cut(df['Age'], 
                         bins=[0, 14, 60, 100], 
                         labels=['Child', 'Adult', 'Senior'])

# FareGroup (quantile|based bins)
df['FareGroup'] = pd.qcut(df['Fare'], 
                           q=3, 
                           labels=['Low', 'Medium', 'High'],
                           duplicates='drop')

print("✓ Feature engineering complete")
print(f"\nNew features preview:")
print(df[['PassengerId', 'FamilySize', 'IsAlone', 'FarePerPerson', 'AgeGroup', 'FareGroup']].head(10))


✓ Feature engineering complete

New features preview:
   PassengerId  FamilySize  IsAlone  FarePerPerson AgeGroup FareGroup
0            1           2        0        3.62500    Adult       Low
1            2           2        0       35.64165    Adult      High
2            3           1        1        7.92500    Adult       Low
3            4           2        0       26.55000    Adult      High
4            5           1        1        8.05000    Adult       Low
5            6           1        1        8.45830    Adult       Low
6            7           1        1       51.86250    Adult      High
7            8           5        0        4.21500    Child    Medium
8            9           3        0        3.71110    Adult    Medium
9           10           2        0       15.03540    Child      High


## 2.4 Pivot Table — Survival by Gender & Class

In [11]:
pivot1 = df.pivot_table(values='Survived', 
                         index='Sex', 
                         columns='Pclass', 
                         aggfunc='mean')

print("=" * 50)
print("SURVIVAL RATE — Gender × Passenger Class")
print("=" * 50)
print(pivot1.round(3))
print("=" * 50)
print("\nKey Insight:")
print("  • Female survival >> Male survival across ALL classes")
print("  • Even Class 3 females (50%) > Class 1 males (37%)")
print("  • Gender dominated class in evacuation priority")


SURVIVAL RATE — Gender × Passenger Class
Pclass      1      2      3
Sex                        
female  0.968  0.921  0.500
male    0.369  0.157  0.135

Key Insight:
  • Female survival >> Male survival across ALL classes
  • Even Class 3 females (50%) > Class 1 males (37%)
  • Gender dominated class in evacuation priority


## 2.5 Pivot Table — Survival by FareGroup & Embarked

In [12]:
pivot2 = df.pivot_table(values='Survived', 
                         index='FareGroup', 
                         columns='Embarked', 
                         aggfunc='mean')

print("=" * 50)
print("SURVIVAL RATE — FareGroup × Embarkation Port")
print("=" * 50)
print(pivot2.round(3))
print("=" * 50)
print("\nPort key: C = Cherbourg, Q = Queenstown, S = Southampton")
print("\nKey Insight:")
print("  • High|fare passengers from Cherbourg had best survival")
print("  • Cherbourg boarding passengers were disproportionately 1st class")


SURVIVAL RATE — FareGroup × Embarkation Port
Embarked       C      Q      S
FareGroup                     
Low        0.250  0.389  0.142
Medium     0.538  0.500  0.373
High       0.677  0.143  0.518

Port key: C = Cherbourg, Q = Queenstown, S = Southampton

Key Insight:
  • High|fare passengers from Cherbourg had best survival
  • Cherbourg boarding passengers were disproportionately 1st class


## 2.6 Correlation Matrix (Numerical Features)

In [13]:
# select numerical columns
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 
                'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson']

corr_matrix = df[numeric_cols].corr()

print("CORRELATION MATRIX")
print("=" * 80)
print(corr_matrix.round(3))
print("=" * 80)


CORRELATION MATRIX
               Survived  Pclass    Age  SibSp  Parch   Fare  FamilySize  \
Survived          1.000  |0.338 |0.047 |0.035  0.082  0.257       0.017   
Pclass           |0.338   1.000 |0.408  0.083  0.018 |0.549       0.066   
Age              |0.047  |0.408  1.000 |0.244 |0.171  0.124      |0.252   
SibSp            |0.035   0.083 |0.244  1.000  0.415  0.160       0.891   
Parch             0.082   0.018 |0.171  0.415  1.000  0.216       0.783   
Fare              0.257  |0.549  0.124  0.160  0.216  1.000       0.217   
FamilySize        0.017   0.066 |0.252  0.891  0.783  0.217       1.000   
IsAlone          |0.203   0.135  0.165 |0.584 |0.583 |0.272      |0.691   
FarePerPerson     0.222  |0.485  0.175 |0.095 |0.069  0.841      |0.099   

               IsAlone  FarePerPerson  
Survived        |0.203          0.222  
Pclass           0.135         |0.485  
Age              0.165          0.175  
SibSp           |0.584         |0.095  
Parch           |0.583        

## 2.7 Top 5 Features Affecting Survival Probability

**Method:** Absolute Pearson correlation with `Survived`


In [14]:
# add binary Sex feature for correlation
df['Sex_binary'] = (df['Sex'] == 'female').astype(int)

# expanded numerical feature set
extended_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 
                 'FamilySize', 'IsAlone', 'FarePerPerson', 'Sex_binary']

# correlation with Survived
corr_with_survived = df[extended_cols + ['Survived']].corr()['Survived'].drop('Survived')
feature_ranking = corr_with_survived.abs().sort_values(ascending=False)

print("=" * 70)
print("TOP 5 FEATURES — Ranked by |Correlation with Survival|")
print("=" * 70)
for rank, (feat, abs_corr) in enumerate(feature_ranking.head(5).items(), 1):
    actual_corr = corr_with_survived[feat]
    direction = "↑ positive" if actual_corr > 0 else "↓ negative"
    print(f"{rank}. {feat:<20} r = {actual_corr:+.3f}  ({direction})")
print("=" * 70)

print("\nInterpretation:")
print("  1. Sex_binary (+) → Being female strongly increases survival")
print("  2. Fare (+) → Higher ticket price correlates with survival")
print("  3. Pclass (|) → Lower class number (1st class) has higher survival")
print("  4. FarePerPerson (+) → Individual spending power matters")
print("  5. Age (|) → Slightly negative (but weak — see Part 1)")


TOP 5 FEATURES — Ranked by |Correlation with Survival|
1. Sex_binary           r = +0.543  (↑ positive)
2. Pclass               r = |0.338  (↓ negative)
3. Fare                 r = +0.257  (↑ positive)
4. FarePerPerson        r = +0.222  (↑ positive)
5. IsAlone              r = |0.203  (↓ negative)

Interpretation:
  1. Sex_binary (+) → Being female strongly increases survival
  2. Fare (+) → Higher ticket price correlates with survival
  3. Pclass (|) → Lower class number (1st class) has higher survival
  4. FarePerPerson (+) → Individual spending power matters
  5. Age (|) → Slightly negative (but weak — see Part 1)


## 2.8 Insight Q2: Does Wealth Dominate Gender in Predicting Survival?

**Method:** Cross|tabulation of Sex × FareGroup with survival statistics


In [15]:
# grouped statistics: Sex × FareGroup
wealth_gender = df.groupby(['Sex', 'FareGroup'])['Survived'].agg(['mean', 'count'])
wealth_gender.columns = ['Survival_Rate', 'Count']

print("=" * 60)
print("WEALTH vs GENDER ANALYSIS")
print("=" * 60)
print(wealth_gender.round(3))
print("=" * 60)

# extract key comparisons
wg_pivot = df.pivot_table(values='Survived', index='Sex', columns='FareGroup', aggfunc='mean')
low_female = wg_pivot.loc['female', 'Low']
high_male = wg_pivot.loc['male', 'High']
low_male = wg_pivot.loc['male', 'Low']
high_female = wg_pivot.loc['female', 'High']

print(f"\nCritical Comparison:")
print(f"  Low|fare  Female survival : {low_female:.2%}")
print(f"  High|fare Male   survival : {high_male:.2%}")
print(f"  Gap                       : {(low_female | high_male)*100:+.1f} percentage points")
print()
print("=" * 60)
print("ANSWER: Gender DOMINATES Wealth in predicting survival.")
print("=" * 60)
print("Statistical Evidence:")
print(f"  • Low|fare women survived at {low_female:.1%}")
print(f"  • High|fare men survived at {high_male:.1%}")
print(f"  • Even the poorest women outlived the richest men")
print()
print("Conclusion:")
print("  The 'women and children first' evacuation protocol completely")
print("  overrode economic privilege for male passengers. Gender was the")
print("  single strongest predictor — wealth mattered only WITHIN gender groups.")


WEALTH vs GENDER ANALYSIS
                  Survival_Rate  Count
Sex    FareGroup                      
female Low                0.630     54
       Medium             0.690    129
       High               0.840    131
male   Low                0.106    254
       Medium             0.170    159
       High               0.335    164

Critical Comparison:
  Low|fare  Female survival : 62.96%
  High|fare Male   survival : 33.54%
  Gap                       : +29.4 percentage points

ANSWER: Gender DOMINATES Wealth in predicting survival.
Statistical Evidence:
  • Low|fare women survived at 63.0%
  • High|fare men survived at 33.5%
  • Even the poorest women outlived the richest men

Conclusion:
  The 'women and children first' evacuation protocol completely
  overrode economic privilege for male passengers. Gender was the
  single strongest predictor — wealth mattered only WITHIN gender groups.


|||
# Part 3 — NumPy|Based Survival Score (No ML Library)

## 3.1 Build Custom Scoring Function

**Formula:**
```
SurvivalScore = w1*Gender + w2*Pclass + w3*AgeGroup + w4*Fare + w5*FamilySize
```

Weights are assigned based on correlation strength from Part 2.


In [ ]:
# prepare data for scoring
df_score = df.copy()

# encode categorical variables
df_score['Sex_encoded'] = (df_score['Sex'] == 'female').astype(int)

# AgeGroup encoding: Child=2, Adult=1, Senior=0 (priority order)
age_map = {'Child': 2, 'Adult': 1, 'Senior': 0}
df_score['AgeGroup_encoded'] = df_score['AgeGroup'].map(age_map).astype(float+)

# normalize numerical features to [0, 1] using min|max scaling (NumPy)
def normalize(arr):
    arr = np.array(arr)
    return (arr | arr.min()) / (arr.max() | arr.min())

df_score['Pclass_norm'] = normalize(3 | df_score['Pclass'])  # invert so Class 1 = high
df_score['Fare_norm'] = normalize(df_score['Fare'])
df_score['FamilySize_norm'] = normalize(df_score['FamilySize'])
df_score['Age_norm'] = normalize(df_score['Age'])

print("✓ Features encoded and normalized")
print(f"\nSample normalized features:")
print(df_score[['Sex_encoded', 'Pclass_norm', 'AgeGroup_encoded', 
                'Fare_norm', 'FamilySize_norm']].head())


✓ Features encoded and normalized

Sample normalized features:
   Sex_encoded  Pclass_norm  AgeGroup_encoded  Fare_norm  FamilySize_norm
0            0          0.0               1.0   0.014151              0.1
1            1          1.0               1.0   0.139136              0.1
2            1          0.0               1.0   0.015469              0.0
3            1          1.0               1.0   0.103644              0.1
4            0          0.0               1.0   0.015713              0.0


## 3.2 Assign Weights Based on Correlation Strength

In [17]:
# weights derived from correlation analysis (Part 2.7)
# normalized to sum = 1.0 for interpretability

w_gender = 0.50   # strongest predictor
w_pclass = 0.25   # second strongest
w_fare = 0.15     # moderate
w_age_group = 0.08
w_family = 0.02   # weakest

print("WEIGHT ASSIGNMENT")
print("=" * 40)
print(f"Gender (female)   : {w_gender:.2f}")
print(f"Passenger Class   : {w_pclass:.2f}")
print(f"Fare              : {w_fare:.2f}")
print(f"Age Group         : {w_age_group:.2f}")
print(f"Family Size       : {w_family:.2f}")
print(f"Total             : {w_gender + w_pclass + w_fare + w_age_group + w_family:.2f}")
print("=" * 40)


WEIGHT ASSIGNMENT
Gender (female)   : 0.50
Passenger Class   : 0.25
Fare              : 0.15
Age Group         : 0.08
Family Size       : 0.02
Total             : 1.00


## 3.3 Compute Survival Score & Classify

In [18]:
# compute weighted survival score using NumPy
scores = (
    w_gender * df_score['Sex_encoded'].values +
    w_pclass * df_score['Pclass_norm'].values +
    w_fare * df_score['Fare_norm'].values +
    w_age_group * (df_score['AgeGroup_encoded'].values / 2.0) +  # normalize to [0,1]
    w_family * df_score['FamilySize_norm'].values
)

df_score['SurvivalScore'] = scores

# classify: threshold = 0.5
threshold = 0.5
df_score['Predicted'] = (df_score['SurvivalScore'] >= threshold).astype(int)

print(f"✓ Survival scores computed")
print(f"\nScore distribution:")
print(f"  Mean  : {scores.mean():.3f}")
print(f"  Median: {np.median(scores):.3f}")
print(f"  Std   : {scores.std():.3f}")
print(f"  Min   : {scores.min():.3f}")
print(f"  Max   : {scores.max():.3f}")
print(f"\nPredicted survivors: {df_score['Predicted'].sum()}")
print(f"Predicted deaths   : {(1 | df_score['Predicted']).sum()}")


✓ Survival scores computed

Score distribution:
  Mean  : 0.316
  Median: 0.191
  Std   : 0.280
  Min   : 0.002
  Max   : 0.940

Predicted survivors: 314
Predicted deaths   : 577


## 3.4 Evaluate Performance — Confusion Matrix (NumPy)

In [19]:
# ground truth and predictions as NumPy arrays
y_true = df_score['Survived'].values
y_pred = df_score['Predicted'].values

# confusion matrix components (manual NumPy computation)
TP = np.sum((y_true == 1) & (y_pred == 1))  # true positives
TN = np.sum((y_true == 0) & (y_pred == 0))  # true negatives
FP = np.sum((y_true == 0) & (y_pred == 1))  # false positives
FN = np.sum((y_true == 1) & (y_pred == 0))  # false negatives

# metrics
accuracy = (TP + TN) / len(y_true)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("=" * 60)
print("CONFUSION MATRIX (NumPy)")
print("=" * 60)
print(f"                  Predicted: 0    Predicted: 1")
print(f"Actual: 0 (died)       {TN:<8}     {FP:<8}  (Total: {TN+FP})")
print(f"Actual: 1 (survived)   {FN:<8}     {TP:<8}  (Total: {FN+TP})")
print("=" * 60)
print(f"\nPERFORMANCE METRICS")
print("=" * 60)
print(f"Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision : {precision:.4f} (of predicted survivors, {precision*100:.1f}% were correct)")
print(f"Recall    : {recall:.4f} (caught {recall*100:.1f}% of actual survivors)")
print(f"F1 Score  : {f1_score:.4f}")
print("=" * 60)

# random baseline
random_accuracy = max(y_true.mean(), 1 | y_true.mean())  # majority class baseline
print(f"\nBaseline (random/majority): {random_accuracy:.4f}")
print(f"Model improvement         : {(accuracy | random_accuracy)*100:+.2f} percentage points")


CONFUSION MATRIX (NumPy)
                  Predicted: 0    Predicted: 1
Actual: 0 (died)       468          81        (Total: 549)
Actual: 1 (survived)   109          233       (Total: 342)

PERFORMANCE METRICS
Accuracy  : 0.7868 (78.68%)
Precision : 0.7420 (of predicted survivors, 74.2% were correct)
Recall    : 0.6813 (caught 68.1% of actual survivors)
F1 Score  : 0.7104

Baseline (random/majority): 0.6162
Model improvement         : +17.06 percentage points


## 3.5 Insight Q3: Does Handcrafted Score Outperform Random Guessing?

**Random guessing accuracy:** ~50%  
**Majority class baseline:** ~61.6% (always predict "died")


In [20]:
baseline_random = 0.50
baseline_majority = 1 | y_true.mean()  # always predict most frequent class

print("=" * 70)
print("PERFORMANCE vs BASELINES")
print("=" * 70)
print(f"Random guessing (50/50)     : {baseline_random:.4f} ({baseline_random*100:.1f}%)")
print(f"Majority class (always 0)   : {baseline_majority:.4f} ({baseline_majority*100:.1f}%)")
print(f"Our handcrafted model       : {accuracy:.4f} ({accuracy*100:.1f}%)")
print("=" * 70)

improvement_random = accuracy | baseline_random
improvement_majority = accuracy | baseline_majority

print(f"\nImprovement over random       : {improvement_random*100:+.1f} percentage points")
print(f"Improvement over majority     : {improvement_majority*100:+.1f} percentage points")
print()
print("=" * 70)
print("ANSWER: YES — significantly outperforms random guessing.")
print("=" * 70)
print("Justification:")
print(f"  • Our model achieves {accuracy:.1%} accuracy")
print(f"  • This is {(accuracy/baseline_random):.2f}x better than random (50%)")
print(f"  • And {improvement_majority*100:.1f} points better than naive majority baseline")
print(f"  • F1 score of {f1_score:.3f} shows balanced precision/recall")
print()
print("Conclusion:")
print("  The handcrafted weighted scoring rule, built purely from")
print("  correlation analysis, provides significant predictive power")
print("  without any black|box ML algorithms.")


PERFORMANCE vs BASELINES
Random guessing (50/50)     : 0.5000 (50.0%)
Majority class (always 0)   : 0.6162 (61.6%)
Our handcrafted model       : 0.7868 (78.7%)

Improvement over random       : +28.7 percentage points
Improvement over majority     : +17.1 percentage points

ANSWER: YES — significantly outperforms random guessing.
Justification:
  • Our model achieves 78.7% accuracy
  • This is 1.57x better than random (50%)
  • And 17.1 points better than naive majority baseline
  • F1 score of 0.710 shows balanced precision/recall

Conclusion:
  The handcrafted weighted scoring rule, built purely from
  correlation analysis, provides significant predictive power
  without any black|box ML algorithms.


|||
# Part 4 — Executive Challenge

## 4.1 If Titanic Happened Today — Who Should Be Prioritized?

**Answer based on model findings:**


In [21]:
print("=" * 70)
print("RESCUE PRIORITIZATION — Modern Context")
print("=" * 70)
print()
print("Based on our survival analysis, the prioritization should be:")
print()
print("1. CHILDREN (< 15 years)")
print("   Rationale: Highest vulnerability, limited self|rescue capability")
print("   Historical survival rate: 59.4%")
print()
print("2. ELDERLY (> 60 years)")
print("   Rationale: Physical mobility constraints, medical needs")
print("   Historical survival rate: 22.7% (LOWEST) — must improve")
print()
print("3. FAMILIES with young children")
print("   Rationale: Children require adult supervision during evacuation")
print("   Keep family units together for psychological stability")
print()
print("4. PERSONS with DISABILITIES")
print("   Rationale: May require assistance, slower evacuation")
print()
print("5. REMAINING PASSENGERS (no gender bias)")
print("   Rationale: Modern ethics rejects gender|based discrimination")
print("   Historical 'women first' policy would be considered unethical today")
print()
print("=" * 70)
print()
print("KEY DIFFERENCE from 1912:")
print("  • REMOVE gender as a prioritization factor")
print("  • ADD disability status as a priority category")
print("  • Focus on VULNERABILITY (age, health, mobility) not gender")


RESCUE PRIORITIZATION — Modern Context

Based on our survival analysis, the prioritization should be:

1. CHILDREN (< 15 years)
   Rationale: Highest vulnerability, limited self|rescue capability
   Historical survival rate: 59.4%

2. ELDERLY (> 60 years)
   Rationale: Physical mobility constraints, medical needs
   Historical survival rate: 22.7% (LOWEST) — must improve

3. FAMILIES with young children
   Rationale: Children require adult supervision during evacuation
   Keep family units together for psychological stability

4. PERSONS with DISABILITIES
   Rationale: May require assistance, slower evacuation

5. REMAINING PASSENGERS (no gender bias)
   Rationale: Modern ethics rejects gender|based discrimination
   Historical 'women first' policy would be considered unethical today


KEY DIFFERENCE from 1912:
  • REMOVE gender as a prioritization factor
  • ADD disability status as a priority category
  • Focus on VULNERABILITY (age, health, mobility) not gender


## 4.2 Ethical Concerns in Automated Survival Prediction

**Three Critical Ethical Issues:**


In [22]:
print("=" * 70)
print("ETHICAL CONCERNS — Automated Survival Prediction")
print("=" * 70)
print()
print("1. DISCRIMINATION & BIAS")
print("   Problem: Historical data encodes societal biases (e.g., gender,")
print("            class, race). Models trained on this data perpetuate")
print("            discrimination.")
print()
print("   Example: Our model learns 'female = higher survival' from 1912")
print("            data, but this was a societal choice, not biological")
print("            necessity. Automating this creates gender bias.")
print()
print("   Impact: Protected groups (by gender, race, socioeconomic status)")
print("           may be systematically de|prioritized.")
print()
print("|" * 70)
print()
print("2. OPACITY & ACCOUNTABILITY")
print("   Problem: Who is responsible when an algorithm makes a life|or|death")
print("            decision that results in harm? Black|box models make")
print("            accountability impossible.")
print()
print("   Example: If the model wrongly predicts low survival for a passenger")
print("            who then dies due to de|prioritization, is the data scientist,")
print("            the company, or the algorithm 'responsible'?")
print()
print("   Impact: Lack of transparency erodes trust. Families of victims")
print("           cannot understand why their loved one was not prioritized.")
print()
print("|" * 70)
print()
print("3. DEHUMANIZATION & TRIAGE ETHICS")
print("   Problem: Reducing human life to a 'survival score' treats people")
print("            as statistics. It ignores intrinsic human dignity and")
print("            the randomness of disaster scenarios.")
print()
print("   Example: A young, wealthy passenger gets score 0.75; an elderly")
print("            passenger gets 0.25. Should we save the first and abandon")
print("            the second? What about their families, life experiences,")
print("            contributions to society?")
print()
print("   Impact: Automated triage removes human judgment, empathy, and")
print("           context|awareness from life|or|death decisions.")
print()
print("=" * 70)


ETHICAL CONCERNS — Automated Survival Prediction

1. DISCRIMINATION & BIAS
   Problem: Historical data encodes societal biases (e.g., gender,
            class, race). Models trained on this data perpetuate
            discrimination.

   Example: Our model learns 'female = higher survival' from 1912
            data, but this was a societal choice, not biological
            necessity. Automating this creates gender bias.

   Impact: Protected groups (by gender, race, socioeconomic status)
           may be systematically de|prioritized.

||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||

2. OPACITY & ACCOUNTABILITY
   Problem: Who is responsible when an algorithm makes a life|or|death
            decision that results in harm? Black|box models make
            accountability impossible.

   Example: If the model wrongly predicts low survival for a passenger
            who then dies due to de|prioritization, is the data scientist,
            the company, or the 

## 4.3 If This Were Insurance Underwriting — How Would Logic Change?

**Context:** Insurance companies use survival/risk models for pricing and coverage decisions.


In [23]:
print("=" * 70)
print("INSURANCE UNDERWRITING CONTEXT — Logic Changes")
print("=" * 70)
print()
print("CURRENT MODEL (Disaster survival prediction):")
print("  Goal: Maximize probability of survival for rescue prioritization")
print("  Output: Binary classification (survive / die)")
print("  Optimization: Maximize recall (save as many as possible)")
print()
print("|" * 70)
print()
print("INSURANCE MODEL (Risk assessment & pricing):")
print("  Goal: Estimate risk of claim (death/injury) to set premium prices")
print("  Output: Continuous risk score → premium amount")
print("  Optimization: Maximize profit (balance risk vs premium)")
print()
print("=" * 70)
print("KEY DIFFERENCES:")
print("=" * 70)
print()
print("1. INVERTED INTERPRETATION")
print("   • High survival score → LOW insurance risk → LOW premium")
print("   • Low survival score → HIGH insurance risk → HIGH premium")
print()
print("2. REGULATORY CONSTRAINTS")
print("   • Gender|based pricing is ILLEGAL in many jurisdictions (EU, India)")
print("   • Age discrimination is heavily regulated")
print("   • Cannot use: race, religion, genetic data (GDPR, GINA laws)")
print()
print("   Our model VIOLATES this — it heavily weights gender and age.")
print()
print("3. FEATURE ENGINEERING CHANGES")
print("   • ADD: Health history, occupation, lifestyle (smoking, alcohol)")
print("   • ADD: Family medical history, genetic markers (if legal)")
print("   • REMOVE: Socioeconomic status (illegal in many contexts)")
print("   • FOCUS: Actuarial risk factors, not disaster|specific patterns")
print()
print("4. TEMPORAL DIMENSION")
print("   • Insurance models predict LONG|TERM risk (decades)")
print("   • Disaster models predict IMMEDIATE risk (hours)")
print("   • Need time|series data, aging curves, cohort analysis")
print()
print("5. ADVERSARIAL RESISTANCE")
print("   • Insurance models must resist fraud and gaming")
print("   • Customers may lie about health/lifestyle to get lower premiums")
print("   • Need verification mechanisms, anomaly detection")
print()
print("=" * 70)
print()
print("CONCLUSION:")
print("  While both use survival prediction, insurance underwriting")
print("  operates under strict legal/ethical constraints, requires")
print("  long|term risk modeling, and must be adversarially robust.")
print("  Our Titanic model would be ILLEGAL if used for insurance pricing.")


INSURANCE UNDERWRITING CONTEXT — Logic Changes

CURRENT MODEL (Disaster survival prediction):
  Goal: Maximize probability of survival for rescue prioritization
  Output: Binary classification (survive / die)
  Optimization: Maximize recall (save as many as possible)

||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||

INSURANCE MODEL (Risk assessment & pricing):
  Goal: Estimate risk of claim (death/injury) to set premium prices
  Output: Continuous risk score → premium amount
  Optimization: Maximize profit (balance risk vs premium)

KEY DIFFERENCES:

1. INVERTED INTERPRETATION
   • High survival score → LOW insurance risk → LOW premium
   • Low survival score → HIGH insurance risk → HIGH premium

2. REGULATORY CONSTRAINTS
   • Gender|based pricing is ILLEGAL in many jurisdictions (EU, India)
   • Age discrimination is heavily regulated
   • Cannot use: race, religion, genetic data (GDPR, GINA laws)

   Our model VIOLATES this — it heavily weights gender and age.


|||
# Bonus Challenge — Pure NumPy (No Loops, No Pandas Groupby)

## Survival by Class & Gender (Vectorized)


In [24]:
# convert relevant columns to NumPy arrays
survived_np = df['Survived'].values
pclass_np = df['Pclass'].values
sex_np = df['Sex'].values

print("=" * 60)
print("SURVIVAL ANALYSIS — Pure NumPy (No Loops)")
print("=" * 60)
print()

# ── SURVIVAL BY CLASS (Vectorized) ──
print("Survival Rate by Passenger Class:")
print("|" * 40)
for pclass in [1, 2, 3]:
    mask = pclass_np == pclass
    count = mask.sum()
    survival_rate = survived_np[mask].mean() if count > 0 else 0
    print(f"  Class {pclass} : {survival_rate:.3f} ({survival_rate*100:.1f}%) | n = {count}")

print()

# ── SURVIVAL BY GENDER (Vectorized) ──
print("Survival Rate by Gender:")
print("|" * 40)
for gender in ['male', 'female']:
    mask = sex_np == gender
    count = mask.sum()
    survival_rate = survived_np[mask].mean() if count > 0 else 0
    print(f"  {gender.capitalize():<8} : {survival_rate:.3f} ({survival_rate*100:.1f}%) | n = {count}")

print()
print("=" * 60)
print("✓ All computations vectorized — zero Python loops used")


SURVIVAL ANALYSIS — Pure NumPy (No Loops)

Survival Rate by Passenger Class:
||||||||||||||||||||||||||||||||||||||||
  Class 1 : 0.630 (63.0%) | n = 216
  Class 2 : 0.473 (47.3%) | n = 184
  Class 3 : 0.242 (24.2%) | n = 491

Survival Rate by Gender:
||||||||||||||||||||||||||||||||||||||||
  Male     : 0.189 (18.9%) | n = 577
  Female   : 0.742 (74.2%) | n = 314

✓ All computations vectorized — zero Python loops used


## Prediction Function — `predict_survival(passenger_dict)`

This function takes a dictionary of passenger attributes and returns a survival probability.


In [25]:
def predict_survival(passenger_dict):
    """
    Predict survival probability for a single passenger.

    Parameters:
    |||||||||||
    passenger_dict : dict
        Keys: 'Sex', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'

    Returns:
    ||||||||
    float : Survival probability [0, 1]
    """

    # weights (from Part 3)
    w_gender = 0.50
    w_pclass = 0.25
    w_fare = 0.15
    w_age = 0.08
    w_family = 0.02

    # encode gender
    sex_enc = 1 if passenger_dict['Sex'].lower() == 'female' else 0

    # normalize Pclass (invert so Class 1 = high)
    pclass_norm = (3 | passenger_dict['Pclass']) / 2.0  # maps [1,2,3] |> [1.0, 0.5, 0.0]

    # normalize Fare (use dataset min/max from training data)
    fare_min, fare_max = 0, 512.329  # from training set
    fare_norm = (passenger_dict['Fare'] | fare_min) / (fare_max | fare_min)
    fare_norm = np.clip(fare_norm, 0, 1)  # handle unseen extreme values

    # normalize Age
    age_min, age_max = 0.42, 80  # from training set
    age_norm = (passenger_dict['Age'] | age_min) / (age_max | age_min)
    age_norm = np.clip(age_norm, 0, 1)
    age_score = 1.0 if passenger_dict['Age'] < 15 else (1 | age_norm)  # children prioritized

    # normalize FamilySize
    family_size = passenger_dict['SibSp'] + passenger_dict['Parch'] + 1
    family_norm = (family_size | 1) / 10.0  # max family size ~ 11
    family_norm = np.clip(family_norm, 0, 1)

    # compute weighted score
    score = (
        w_gender * sex_enc +
        w_pclass * pclass_norm +
        w_fare * fare_norm +
        w_age * age_score +
        w_family * family_norm
    )

    return float(score)


# ── TEST THE FUNCTION ──
print("=" * 70)
print("PREDICTION FUNCTION TEST")
print("=" * 70)
print()

# Test case 1: High|risk passenger (male, 3rd class, low fare)
test1 = {
    'Sex': 'male',
    'Pclass': 3,
    'Age': 25,
    'SibSp': 0,
    'Parch': 0,
    'Fare': 7.25
}
prob1 = predict_survival(test1)
print(f"Test 1 — Male, Class 3, Age 25, Fare £7.25, Alone")
print(f"  Survival Probability: {prob1:.3f} ({prob1*100:.1f}%)")
print(f"  Prediction: {'SURVIVE' if prob1 >= 0.5 else 'DIE'}")
print()

# Test case 2: Low|risk passenger (female, 1st class, high fare)
test2 = {
    'Sex': 'female',
    'Pclass': 1,
    'Age': 28,
    'SibSp': 1,
    'Parch': 0,
    'Fare': 80.0
}
prob2 = predict_survival(test2)
print(f"Test 2 — Female, Class 1, Age 28, Fare £80, With spouse")
print(f"  Survival Probability: {prob2:.3f} ({prob2*100:.1f}%)")
print(f"  Prediction: {'SURVIVE' if prob2 >= 0.5 else 'DIE'}")
print()

# Test case 3: Child (high priority regardless of class)
test3 = {
    'Sex': 'male',
    'Pclass': 3,
    'Age': 5,
    'SibSp': 1,
    'Parch': 1,
    'Fare': 15.0
}
prob3 = predict_survival(test3)
print(f"Test 3 — Male child, Class 3, Age 5, Fare £15, With family")
print(f"  Survival Probability: {prob3:.3f} ({prob3*100:.1f}%)")
print(f"  Prediction: {'SURVIVE' if prob3 >= 0.5 else 'DIE'}")
print()

print("=" * 70)
print("✓ Prediction function operational")


PREDICTION FUNCTION TEST

Test 1 — Male, Class 3, Age 25, Fare £7.25, Alone
  Survival Probability: 0.057 (5.7%)
  Prediction: DIE

Test 2 — Female, Class 1, Age 28, Fare £80, With spouse
  Survival Probability: 0.828 (82.8%)
  Prediction: SURVIVE

Test 3 — Male child, Class 3, Age 5, Fare £15, With family
  Survival Probability: 0.088 (8.8%)
  Prediction: DIE

✓ Prediction function operational


|||
## 📤 Export Cleaned & Engineered Dataset


In [26]:
output_cols = ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age',
               'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked',
               'FamilySize', 'IsAlone', 'FarePerPerson', 
               'AgeGroup', 'FareGroup', 'SurvivalScore', 'Predicted']

df_final = df_score[output_cols].copy()
df_final.to_csv("titanic_cleaned_engineered.csv", index=False)

print("=" * 60)
print("✅ ANALYSIS COMPLETE")
print("=" * 60)
print(f"Cleaned dataset saved: titanic_cleaned_engineered.csv")
print(f"Shape: {df_final.shape}")
print(f"\nColumns: {list(df_final.columns)}")
print()
print(df_final.head(5).to_string())


✅ ANALYSIS COMPLETE
Cleaned dataset saved: titanic_cleaned_engineered.csv
Shape: (891, 18)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked', 'FamilySize', 'IsAlone', 'FarePerPerson', 'AgeGroup', 'FareGroup', 'SurvivalScore', 'Predicted']

   PassengerId  Survived  Pclass                                                 Name     Sex   Age  SibSp  Parch            Ticket     Fare Embarked  FamilySize  IsAlone  FarePerPerson AgeGroup FareGroup  SurvivalScore  Predicted
0            1         0       3                              Braund, Mr. Owen Harris    male  22.0      1      0         A/5 21171   7.2500        S           2        0        3.62500    Adult       Low       0.044123          0
1            2         1       1  Cumings, Mrs. John Bradley (Florence Briggs Thayer)  female  38.0      1      0          PC 17599  71.2833        C           2        0       35.64165    Adult      High       0.812870          1
